# Lab 4, Notebook A — Filtering the spectrum

*Linear Algebra for Data Science, UCU 2026. Uses Session 5.2 (SVD, Eckart–Young, truncated SVD, filter factors) and, for A3, the convolution theorem for the discrete Fourier transform.*

An image is a matrix, and the SVD sorts it into rank-one layers by importance. In this notebook you will keep, drop or shrink those layers for three different purposes: to **compress** an image (A1), to remove **noise** from it (A2), and to undo a **blur** (A3). The same idea runs through all three: write the problem in a basis where it is diagonal, then decide what to do with each coordinate. The last part also shows why a mathematically exact inverse can be numerically worthless.

**How to work.** Most tasks ask you to **predict first, in writing, then compute, then explain any difference**. Write the prediction *before* running the cell. A wrong prediction with an honest explanation earns more at the defence than a correct number you cannot explain.

**Rules.**
- Do not call `np.linalg.inv` or `np.linalg.pinv` anywhere.
- Never form the blur matrix of the full image (you will see in A3 why you cannot). The only explicit circulant matrix allowed is the small 1-D one in A3(a).
- You may use `np.linalg.svd`, `np.linalg.norm`, `scipy.linalg.qr` and `np.fft`. Library shortcuts such as `scipy.linalg.circulant` are allowed only where a cell says **check**.
- Do not change the seeds.
- Before you submit, run `Kernel → Restart & Run All`.

**Marking.** Core A is worth 2 points at the defence; the code point and the extension are shared with Notebook B.

**Contents.** A1 Compression and Eckart–Young · A2 Denoising · A3 Deblurring · Extension E1 (choosing λ without knowing the noise). Attempt **at most one** extension in the whole lab.

| | |
|---|---|
| **Team member 1** | *name, surname* |
| **Team member 2** | *name, surname* |
| **Extension attempted** (at most one in the whole lab) | *none / E1 / E2* |
| **Late days used on this lab** | *0* |
| **Runs top to bottom from a clean kernel** | *yes / no* |

In [ ]:
import sys, time
import numpy as np
import scipy, scipy.linalg as sl
import matplotlib, matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_sample_image

print("python", sys.version.split()[0], "| numpy", np.__version__, "| scipy", scipy.__version__,
      "| matplotlib", matplotlib.__version__, "| sklearn", sklearn.__version__)
np.set_printoptions(precision=4, suppress=False, linewidth=110)

img = load_sample_image("flower.jpg").astype(float).mean(axis=2) / 255   # grey levels in [0, 1]
m, n = img.shape
rel = lambda X: np.linalg.norm(X - img) / np.linalg.norm(img)            # relative Frobenius error
print(f"image: {m} x {n} = {m*n} pixels")
plt.figure(figsize=(5, 3.4)); plt.imshow(img, cmap="gray", vmin=0, vmax=1); plt.axis("off"); plt.show()

---
## A1. Compression and Eckart–Young

Write the SVD as $A = \sum_j \sigma_j \mathbf{u}_j\mathbf{v}_j^\top$. The truncated SVD $A_k = \sum_{j\le k}\sigma_j\mathbf{u}_j\mathbf{v}_j^\top$ is stored as $k$ columns of $U$, $k$ rows of $V^\top$ and $k$ singular values.

**(a)** Implement `rank_k(U, s, Vt, k)`, which returns $A_k$ from the thin SVD. Use broadcasting, not `np.diag`, and do not multiply by any zero columns.

In [ ]:
U, s, Vt = np.linalg.svd(img, full_matrices=False)

def rank_k(U, s, Vt, k):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check
assert np.allclose(rank_k(U, s, Vt, len(s)), img)
assert np.linalg.matrix_rank(rank_k(U, s, Vt, 7)) == 7
plt.figure(figsize=(5, 3.2))
plt.semilogy(np.arange(1, len(s) + 1), s / s[0])
plt.xlabel("j"); plt.ylabel(r"$\sigma_j/\sigma_1$"); plt.title("singular values of the image"); plt.tight_layout(); plt.show()
print(f"sigma_1 = {s[0]:.1f}, sigma_{len(s)} = {s[-1]:.2e}")

**(b)** *Predict, then compute.* Implement two functions that use **only the singular values**:
- `predicted_errors(s, k)` returns the relative Frobenius error $\|A-A_k\|_F/\|A\|_F$ and the 2-norm error $\|A-A_k\|_2$ of the best rank-$k$ approximation;
- `storage(k, m, n)` returns the fraction of the $mn$ pixel values needed to store $A_k$ in factored form.

Before running the comparison below: will the actual errors of `rank_k` **equal** your predictions, or only be bounded by them? Name the theorem.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
def predicted_errors(s, k):
    # YOUR CODE HERE
    raise NotImplementedError

def storage(k, m, n):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check
print("   k | predicted rel. F  actual rel. F | predicted 2-norm  actual 2-norm | storage")
for k in [5, 10, 20, 50, 100]:
    Ak = rank_k(U, s, Vt, k)
    pf, p2 = predicted_errors(s, k)
    af, a2 = rel(Ak), np.linalg.norm(img - Ak, 2)
    print(f"{k:4d} | {pf:16.4f} {af:14.4f} | {p2:16.3f} {a2:14.3f} | {storage(k, m, n):6.2%}")
    assert np.isclose(pf, af) and np.isclose(p2, a2)

**Your answer.** *Write it here.*

**(c)** From the singular values alone, find the smallest $k$ with relative Frobenius error at most 10%, 5% and 2%, and the storage each needs. Then find the **break-even rank**, above which the factored form needs more storage than the image itself. Is this image "low-rank"? Compare with a JPEG file, which stores a typical photograph in roughly 5–10% of its raw size with no visible loss.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(d)** *A competitor.* Instead of singular vectors, use $k$ **actual columns** of the image: choose them by QR with column pivoting, $AP = QR$, and project the whole image onto the span of the first $k$ columns of $Q$. Implement `column_subset_approx(A, k)`, returning the approximation and the indices of the chosen columns.

*Predict before running:* at $k = 20$, will its error be smaller, equal or larger than that of $A_{20}$? Is there any $k$ at which it could win?

**Your prediction** *(write it before running the next cell)*:

In [ ]:
def column_subset_approx(A, k):
    # YOUR CODE HERE
    raise NotImplementedError

k = 20
Ac, cols = column_subset_approx(img, k)
print(f"k = {k}: column subset error {rel(Ac):.4f}, truncated SVD error {rel(rank_k(U, s, Vt, k)):.4f}")
print("chosen columns:", np.sort(cols))

**Your answer.** *Write it here.*

**(e)** Look at the approximations below. Where in the image is the error at $k = 30$ concentrated? Explain why, using the fact that each layer $\sigma_j\mathbf{u}_j\mathbf{v}_j^\top$ is an outer product.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(9, 6.2))
for a, k in zip(ax.flat, [None, 10, 30, 70]):
    X = img if k is None else rank_k(U, s, Vt, k)
    a.imshow(np.clip(X, 0, 1), cmap="gray", vmin=0, vmax=1); a.axis("off")
    a.set_title("original" if k is None else f"k = {k}, error {rel(X):.3f}, storage {storage(k, m, n):.0%}")
plt.tight_layout(); plt.show()
plt.figure(figsize=(5, 3.4)); plt.imshow(np.abs(img - rank_k(U, s, Vt, 30)), cmap="magma"); plt.axis("off")
plt.title("|A - A_30|"); plt.colorbar(); plt.show()

**Your answer.** *Write it here.*

---
## A2. Denoising

Now we observe $Y = A + E$, where the entries of $E$ are independent $\mathcal N(0, \text{sd}^2)$. Two facts, stated without proof:

- **Weyl's inequality** for singular values: $|\sigma_j(A+E) - \sigma_j(A)| \le \|E\|_2$ for every $j$.
- **Noise floor.** For an $m\times n$ matrix with independent $\mathcal N(0,\text{sd}^2)$ entries and $m, n$ large, $\|E\|_2 \approx \text{sd}\,(\sqrt m + \sqrt n)$.

So singular values of $A$ well below the floor are drowned by the noise, and those well above it survive with a small shift. This suggests keeping only the singular values of $Y$ that stand above the floor.

In [ ]:
sd = 0.05
E = sd * np.random.default_rng(4).standard_normal((m, n))
Y = img + E
print(f"sd = {sd}: error of the noisy image {rel(Y):.4f}")

**(a)** *Predict, then compute.* Predict $\|E\|_2$ from the noise-floor formula. Then compute it, and check Weyl's inequality: compute $\max_j|\sigma_j(Y) - \sigma_j(A)|$ and compare it with $\|E\|_2$.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(b)** Plot the singular values of $A$ and of $Y$ together with the floor. Then compare two choices of the rank for the truncated SVD of $Y$:
- the **threshold rule** $k_{\text{thr}} = \#\{j : \sigma_j(Y) > \text{floor}\}$;
- the **oracle** $k_{\text{or}}$, the rank that minimises the error against the clean image, found by trying $k = 1,\dots,200$.

Report both ranks and errors. Why can the oracle not be used in practice?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(c)** Truncation keeps a layer completely or drops it completely. A softer choice multiplies layer $j$ by a **filter factor**
$$f_j = \frac{\sigma_j^2}{\sigma_j^2 + \lambda},$$
the same factors as in Tikhonov regularisation (frame `tikhonov-filter`). Implement `filter_factors(s, lam)` and `spectral_filter(U, s, Vt, f)`, which returns $\sum_j f_j\sigma_j\mathbf{u}_j\mathbf{v}_j^\top$. Use $\lambda = \text{floor}^2$: what is $f_j$ for a singular value exactly at the floor? Compare the error with (b).

In [ ]:
def filter_factors(s, lam):
    # YOUR CODE HERE
    raise NotImplementedError

def spectral_filter(U, s, Vt, f):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check
assert np.allclose(spectral_filter(U, s, Vt, np.ones_like(s)), img)
f = filter_factors(sY, floor ** 2)
print(f"soft filter, lambda = floor^2: error {rel(spectral_filter(UY, sY, VYt, f)):.4f}; "
      f"effective rank sum(f) = {f.sum():.1f}")

**Your answer.** *Write it here.*

**(d)** *Predict, then compute.* Double the noise, $\text{sd} = 0.1$ (same seed). Will the oracle rank go up or down, and roughly by how much? Then repeat (a)–(c) for $\text{sd} = 0.1$. Write a function `denoise_report(sd)` that prints the floor, $\|E\|_2$, the noisy error, both ranks with their errors, and the soft-filter error; run it for both noise levels.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
def denoise_report(sd):
    # YOUR CODE HERE
    raise NotImplementedError

for sd_ in [0.05, 0.1]:
    denoise_report(sd_)

**Your answer.** *Write it here.*

**(e)** At $\text{sd} = 0.05$ the best truncation reduces the error from about 16% to about 10%. Why is the gain so modest? Give two reasons, one about the spectrum you plotted in A1(a) and one about the noise that remains in the kept layers. (*Hint:* how many numbers does a rank-$k$ matrix have, and how much noise lands in each?)

**Your answer.** *Write it here.*

---
## A3. Deblurring

A camera that is slightly out of focus replaces each pixel by a weighted average of its neighbours. With **periodic** boundary conditions (the image wraps around at the edges) this is a 2-D **circular convolution** with a kernel $c$, here a Gaussian of width $w = 2$ pixels:
$$b = c \circledast x + e, \qquad e \sim \mathcal N(0, \eta^2 I).$$
As a matrix acting on the vectorised image, the blur $C$ is $mn \times mn$ and **block circulant with circulant blocks**. The convolution theorem says $\operatorname{fft2}(c \circledast x) = \operatorname{fft2}(c)\cdot\operatorname{fft2}(x)$ entrywise, so $C = F^{-1}\operatorname{diag}(\hat c)F$, where $F$ is the 2-D DFT and $\hat c = \operatorname{fft2}(c)$. Since $F/\sqrt{mn}$ is unitary, the singular values of $C$ are the numbers $|\hat c_j|$ and the singular vectors are Fourier modes. **The SVD of the blur costs one FFT.**

In [ ]:
def kernel(m, n, w):
    # periodic Gaussian blur kernel centred at pixel (0, 0), entries sum to 1
    i = np.minimum(np.arange(m), m - np.arange(m))
    j = np.minimum(np.arange(n), n - np.arange(n))
    K = np.exp(-(i[:, None] ** 2 + j[None, :] ** 2) / (2 * w ** 2))
    return K / K.sum()

w = 2.0
kh = np.fft.fft2(kernel(m, n, w))                                 # the "singular values", with phases
blur = lambda X: np.real(np.fft.ifft2(np.fft.fft2(X) * kh))       # X -> C X, never forming C

**(a)** *The 1-D version, explicitly.* For a vector $c\in\mathbb R^N$ the circulant matrix has entries $C_{ij} = c_{(i-j) \bmod N}$, so that $C\mathbf x = c\circledast\mathbf x$. Implement `circulant(c)`, **check** it against `scipy.linalg.circulant`, and check that $C\mathbf x$ equals the FFT formula. Then confirm that the singular values of $C$ for the 64-point kernel below are the numbers $|\hat c_j|$, up to order.

In [ ]:
def circulant(c):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check
c1 = kernel(64, 1, w)[:, 0]
C1 = circulant(c1)
x1 = np.random.default_rng(0).standard_normal(64)
assert np.allclose(C1, sl.circulant(c1))
assert np.allclose(C1 @ x1, np.real(np.fft.ifft(np.fft.fft(c1) * np.fft.fft(x1))))
sv = np.sort(np.linalg.svd(C1, compute_uv=False))
print("singular values of C equal |fft(c)|:", np.allclose(sv, np.sort(np.abs(np.fft.fft(c1)))))
print("largest and smallest:", sv[-1], sv[0])

**(b)** *Cost.* Compute the memory, in GB, needed to store $C$ for our image as a dense `float64` matrix, and the order of magnitude of the flop count of its SVD ($\approx (mn)^3$ up to a modest constant). Compare with the cost of one 2-D FFT, about $5\,mn\log_2(mn)$ flops. What is the largest image whose $C$ fits into 16 GB of memory?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(c)** *Predict, then compute.* The blurred image is observed with noise of size $\eta = 10^{-3}$ per pixel, a quarter of one grey level in an 8-bit image. Compute the condition number $\kappa(C) = \max|\hat c_j| / \min|\hat c_j|$. Then predict:
1. the size of the error of the **naive inverse** $x = C^{-1}b$ (divide by $\hat c$ in Fourier space);
2. the error of the naive inverse applied to the blurred image **without** noise.

Then compute both, together with the error of the blurred image itself.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
eta = 1e-3
noise = np.random.default_rng(11).standard_normal((m, n))     # one fixed draw, scaled by eta below
B0 = blur(img)                                                  # noise-free blurred image
B = B0 + eta * noise
print(f"eta = {eta}: error of the blurred image {rel(B):.4f}")

In [ ]:
def deblur_naive(B, kh):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE: condition number, and the errors of deblur_naive on B and on B0

**Your answer.** *Write it here.*

**(d)** Implement two regularised inverses, both in Fourier space:
- `deblur_tikhonov(B, kh, lam)` with filter factors $f_j = |\hat c_j|^2/(|\hat c_j|^2+\lambda)$, that is $\hat x = \overline{\hat c}\,\hat b/(|\hat c|^2+\lambda)$;
- `deblur_tsvd(B, kh, tau)`, the truncated SVD: keep $\hat b_j/\hat c_j$ where $|\hat c_j| > \tau$ and set the rest to zero.

Sweep $\lambda$ and $\tau$ over the grids below and plot both errors on one axis against $\lambda$ and $\tau^2$. Report the best value of each (the oracle) and the fraction of frequencies the best truncation keeps. Explain from the filter factors why the best $\tau$ is close to $\sqrt\lambda$.

In [ ]:
def deblur_tikhonov(B, kh, lam):
    # YOUR CODE HERE
    raise NotImplementedError

def deblur_tsvd(B, kh, tau):
    # YOUR CODE HERE
    raise NotImplementedError

lams = 10.0 ** np.arange(-7, 0.01, 0.25)
taus = np.sqrt(lams)
# YOUR CODE HERE: errors over the grids, the two oracles, and the plot

**Your answer.** *Write it here.*

**(e)** *Choosing $\lambda$ without the truth.* The true image leaves the residual $\|Cx - b\| = \|e\| \approx \eta\sqrt{mn}$. The **discrepancy principle** asks the same of the reconstruction: choose the largest $\lambda$ on the grid with $\|Cx_\lambda - b\| \le \eta\sqrt{mn}$. Explain why it would be a bad idea to ask for a much smaller residual.

Implement `discrepancy_lambda(B, kh, eta, lams)`. Compute the residual in Fourier space: by Parseval, $\|r\| = \|\hat r\|/\sqrt{mn}$ with $\hat r = (1 - f)\,\hat b$. Then, for $\eta = 10^{-3}$ and $\eta = 10^{-2}$ (same `noise` draw), report the oracle $\lambda$ and the discrepancy $\lambda$ with their errors. How much do you lose by not knowing the true image?

In [ ]:
def discrepancy_lambda(B, kh, eta, lams):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE: the comparison table for eta = 1e-3 and 1e-2

**Your answer.** *Write it here.*

**(f)** Show a crop of the true image, the blurred noisy image, the naive inverse, and the discrepancy-principle reconstruction, for $\eta = 10^{-3}$. Which details come back, and which do not? Relate what you see to the frequencies that the filter removes.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

---
# Extension — attempt **at most one** in the whole lab

End your extension with a paragraph headed **Finding**: one claim you can defend at the defence, supported by a number in this notebook.

## E1. Choosing λ when the noise level is unknown

*This topic was not lectured; the introduction below is all you need.*

The discrepancy principle needs $\eta$. **Generalised cross-validation** (GCV) does not. Write the Tikhonov reconstruction as $x_\lambda = C(\lambda)b$ and the fitted data as $Cx_\lambda = H_\lambda b$ with the *influence matrix* $H_\lambda = C(C^\top C + \lambda I)^{-1}C^\top$. GCV chooses the $\lambda$ that minimises
$$G(\lambda) = \frac{\frac1N\|(I - H_\lambda)b\|^2}{\left(\frac1N\operatorname{trace}(I - H_\lambda)\right)^2}, \qquad N = mn.$$
It approximates leave-one-out cross-validation: how well does the model predict a pixel it was not fitted to? The denominator penalises small $\lambda$, where $H_\lambda \approx I$ and the fit simply reproduces the noise.

**(a)** *Show* that the eigenvalues of $H_\lambda$ are the filter factors $f_j = |\hat c_j|^2/(|\hat c_j|^2+\lambda)$, so that $\operatorname{trace}(I - H_\lambda) = N - \sum_j f_j$, and that $\|(I-H_\lambda)b\| = \|(1-f)\hat b\|/\sqrt N$. Interpret $\sum_j f_j$ as an "effective number of parameters".

**Your answer.** *Write it here.*

**(b)** Implement `gcv(B, kh, lam)` and choose $\lambda$ by minimising it over `lams`, for $\eta = 10^{-3}$ and $10^{-2}$. Compare the GCV choice with the oracle and with the discrepancy principle, and plot $G(\lambda)$ together with the error curve. Then estimate the noise level from the GCV fit, $\hat\eta^2 = \|(I - H_\lambda)b\|^2/\operatorname{trace}(I-H_\lambda)$, and compare it with the true $\eta$.

In [ ]:
def gcv(B, kh, lam):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE

**Your answer.** *Write it here.*

---
**Before you submit:** run `Kernel → Restart & Run All`, check that every written answer is filled in, and make sure your extension (if any) ends with a **Finding**.